# 04. Оценка сохранённых экспериментов
Оценка читает snapshot данных, конфигурацию и train-fitted нормализатор. Повторного fit нет. Отсутствующие веса не подменяются случайными.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
import json
import pandas as pd
from tsprep.study import validate_completed_study
# Explicit historical study; use a new v3 directory and False for new results.
STUDY_DIR = ROOT / "results" / "study_20261103T184252995031Z"
ALLOW_HISTORICAL_REEVALUATION = True
runs = validate_completed_study(STUDY_DIR, allow_legacy=ALLOW_HISTORICAL_REEVALUATION)
print(STUDY_DIR, "historical reevaluation:", ALLOW_HISTORICAL_REEVALUATION)
runs.groupby(["dataset", "model"])[["mse", "forecast_mae", "forecast_vs_mean", "mse_vs_pca"]].agg(["mean", "std"])


In [ ]:
from tsprep.experiment import load_artifact, evaluate_artifact
# Explicit example, not selected by the test score.
row = runs[(runs.dataset == "aapl") & (runs.model == "tcn_ae")].iloc[0]
ARTIFACT = STUDY_DIR / row.artifact
metrics = evaluate_artifact(ARTIFACT, device="cpu", allow_source_mismatch=ALLOW_HISTORICAL_REEVALUATION)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
assert np.isclose(metrics["mse"], row.mse, rtol=1e-4, atol=1e-6)


In [ ]:
from tsprep.evaluation.reconstruction import collect_embeddings
from tsprep.data_preprocessing.datasets import build_eval_loader
from tsprep.evaluation.visualize import plot_loss_curves, plot_reconstruction_examples, plot_latent_projection
from tsprep.evaluation.latent_space import pca_project
from tsprep.mlcore.trainer import History
model, prepared, exp, manifest, arrays = load_artifact(ARTIFACT, allow_source_mismatch=ALLOW_HISTORICAL_REEVALUATION)
plot_loss_curves(History(**manifest["history"]), title=exp.model.model_name)
out = collect_embeddings(model, build_eval_loader(prepared.test_windows, exp.data.batch_size))
plot_reconstruction_examples(out["x"], out["x_hat"])
labels = None if prepared.regime is None else prepared.regime[prepared.window_starts["test"] + exp.data.window_len - 1]
plot_latent_projection(pca_project(out["z"]), labels, title="Descriptive PCA of embeddings")


## Интерпретация
- Сравнивайте с PCA, raw-window Ridge, train mean/median.
- При одном классе sensitivity/ROC-AUC ограничены или не определены.
- Silhouette с истинными метками не подменяется KMeans.
- Отношение дисперсий зависит от масштаба z и не измеряет информацию.
- Порог аномалий фиксируется по validation; отдельно оцениваются окна и их концы.
- USREC — ретроспективная метка периода.
- Clean reconstruction оценивает шумоподавление отдельно от noisy reconstruction.
